# 02 — Identidad y reloj PDM

Preparado para la fase 2; **sin ejecutar**. El notebook valida el manifiesto y el par BIT/HWH PDM candidato por separado de la ruta de carga XADC. `PROGRAM_PDM_CANDIDATE` y `RUN_CLOCK_EXERCISE` permanecen en `False`; no se programa la FPGA por defecto.

La referencia aceptada sigue siendo XADC. La captura de las cuatro señales `DAT` se implementará y comprobará en fase 3.


## Antes de usar el hardware

No conectes todavía los micrófonos. Te avisaré antes de la prueba física. La secuencia futura será: validar identidad y reloj del candidato, revisar el montaje sin alimentación, medir el rail y después hacer la prueba con los cuatro módulos.

Cableado previsto cuando lo indiquemos (Pmod B, numeración física del conector): `pin 1 CLK` a los cuatro `CLK`; `pin 2 DATA0` a Mic 0 `DAT`; `pin 3 DATA1` a Mic 1 `DAT`; `pin 4 DATA2` a Mic 2 `DAT`; `pin 7 DATA3` a Mic 3 `DAT`; tierra común en `pin 5` o `11`; 3,3 V en `pin 6` o `12`; los cuatro `SEL` a GND. Las cuatro salidas `DAT` van separadas. No usar 5 V ni conectar el Arduino como reloj o como fuente en paralelo.

La asignación física se comprobará de nuevo antes de cablear. El estado `CLOCK_READY` es interno y no mide la forma de onda real en Pmod B.


## 1. Parámetros de la ejecución

Coloca el manifiesto, BIT y HWH del candidato revisado en la ruta indicada. El loader comprueba los SHA-256 fijados, el BUILD_ID del manifiesto y la instancia/dirección del IP en el HWH. Para identificar el paquete instalado, completa exactamente uno de `EXPECTED_SW_COMMIT` (instalación desde Git) o `EXPECTED_WHEEL_SHA256` (instalación desde el wheel preparado).


In [ ]:
from pathlib import Path

PDM_MANIFEST_PATH = Path("evidence/phase-02-closure/reset-correction/candidate/candidate-manifest.json")
EXPECTED_SW_COMMIT = ""  # SHA completo si se instaló desde Git
EXPECTED_WHEEL_SHA256 = ""  # SHA-256 si se instaló el wheel local
EXPECTED_BUILD_ID = "710bf4adefe94edab8e0ab42a5138648"  # candidato PDM fijado por el loader
PDM_IP_NAME = "acoustic_capture_0"
CONFIRMED_BOARD = None  # escribir "pynq_z2" solo al autorizar la carga en la placa
PROGRAM_PDM_CANDIDATE = False
RUN_CLOCK_EXERCISE = False
CLOCK_CYCLES = 10
CLOCK_READY_TIMEOUT_S = 1.0


## 2. Verificar candidato y, si se indica, cargarlo

La validación local comprueba tamaño y SHA-256 de BIT/HWH, BUILD_ID, IP y dirección antes de cualquier interacción con PYNQ. Con los valores por defecto solo valida los archivos; no programa la FPGA ni lee MMIO. Para programar será necesario habilitar expresamente `PROGRAM_PDM_CANDIDATE` y confirmar `pynq_z2`. Después se valida el BUILD_ID de hardware antes de habilitar escrituras del reloj.


In [ ]:
from datetime import datetime, timezone
from importlib import metadata
import json
import re
import sys
import uuid

import pynq_acoustic_camera as camera
from pynq_acoustic_camera import PdmCandidateLoader, PdmClockControl

assert re.fullmatch(r"[0-9a-fA-F]{32}", EXPECTED_BUILD_ID), "BUILD_ID esperado inválido"
commit_pin = bool(EXPECTED_SW_COMMIT)
wheel_pin = bool(EXPECTED_WHEEL_SHA256)
assert commit_pin != wheel_pin, "Completar exactamente una identidad: commit Git o hash del wheel"
if commit_pin:
    assert re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SW_COMMIT), "SHA de commit inválido"
if wheel_pin:
    assert re.fullmatch(r"[0-9a-f]{64}", EXPECTED_WHEEL_SHA256), "SHA-256 del wheel inválido"
dist = metadata.distribution("pynq-acoustic-camera")
direct_url = json.loads(dist.read_text("direct_url.json") or "{}")
installed_commit = direct_url.get("vcs_info", {}).get("commit_id")
archive_info = direct_url.get("archive_info", {})
installed_wheel_sha256 = archive_info.get("hashes", {}).get("sha256")
legacy_hash = archive_info.get("hash", "")
if installed_wheel_sha256 is None and legacy_hash.startswith("sha256="):
    installed_wheel_sha256 = legacy_hash.split("=", 1)[1]
if commit_pin:
    assert installed_commit == EXPECTED_SW_COMMIT, "El commit instalado no coincide con la entrega SW esperada"
else:
    assert installed_wheel_sha256 == EXPECTED_WHEEL_SHA256, "El wheel instalado no coincide con el artefacto esperado"
assert Path(camera.__file__).resolve() == Path(dist.locate_file("pynq_acoustic_camera/__init__.py")).resolve(), "Importar desde la distribución instalada, no desde el checkout"

candidate = PdmCandidateLoader.prepare(PDM_MANIFEST_PATH)
assert candidate.build_id == EXPECTED_BUILD_ID.lower(), "El candidato no tiene el BUILD_ID esperado"
assert candidate.ip_instance == PDM_IP_NAME, "La instancia PDM del HWH no coincide"
print(json.dumps({
    "candidate_manifest": str(candidate.manifest_path),
    "bit_sha256": candidate.bit_sha256,
    "hwh_sha256": candidate.hwh_sha256,
    "build_id_128": candidate.build_id,
    "control_ip_instance": candidate.ip_instance,
    "control_ip_base_address": "0x{:08X}".format(candidate.ip_base_address),
    "program_requested": PROGRAM_PDM_CANDIDATE,
}, indent=2))

PDM_OVERLAY = None
if PROGRAM_PDM_CANDIDATE:
    assert CONFIRMED_BOARD == "pynq_z2", "Confirmar explícitamente CONFIRMED_BOARD='pynq_z2'"
    PDM_OVERLAY = candidate.program_overlay(confirmed_board=CONFIRMED_BOARD)
    assert callable(getattr(PDM_OVERLAY, "is_loaded", None)) and PDM_OVERLAY.is_loaded(), "Overlay no cargado"
    assert PDM_IP_NAME in PDM_OVERLAY.ip_dict, "El overlay no contiene el IP PDM esperado"

    pdm_ip = getattr(PDM_OVERLAY, PDM_IP_NAME)
    pdm = PdmClockControl(pdm_ip.mmio, expected_build_id=EXPECTED_BUILD_ID)
    assert not pdm.clock_enabled, "El reloj debe estar deshabilitado tras reset; detenerse y revisar"

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    run_dir = Path.cwd() / "evidence" / ("02_clock_control-" + run_id)
    run_dir.mkdir(parents=True, exist_ok=False)
    report = {
        "schema_version": 1,
        "milestone": "phase-2-clock-control",
        "run_id": run_id,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.executable,
        "software": {"version": dist.version, "commit": installed_commit,
                     "wheel_sha256": installed_wheel_sha256},
        "candidate": {
            "bit_sha256": candidate.bit_sha256,
            "hwh_sha256": candidate.hwh_sha256,
            "build_id_128": candidate.build_id,
        },
        "hardware_identity": {
            "magic": "0x{:08X}".format(pdm.identity.magic),
            "abi_version": "0x{:08X}".format(pdm.identity.abi_version),
            "capabilities": "0x{:08X}".format(pdm.identity.capabilities),
            "channels": pdm.identity.channels,
            "build_id_128": pdm.identity.build_id,
        },
        "clock": {
            "enabled_after_reset": pdm.clock_enabled,
            "ready_after_reset": pdm.clock_ready,
            "nominal_hz": pdm.pdm_clock_hz,
            "warmup_cycles": pdm.warmup_cycles,
        },
        "clock_exercise_requested": RUN_CLOCK_EXERCISE,
        "physical_clock_measurement": "not_recorded_by_internal_status",
        "microphone_capture": "not_implemented_in_phase_2",
    }
else:
    print("Candidato validado localmente; FPGA sin programar y sin lectura MMIO.")


## 3. Habilitación y comprobación de estado

Esta celda escribe `CLOCK_ENABLE` únicamente si `RUN_CLOCK_EXERCISE=True`. Dejarlo en `False` hasta que se haya revisado la identidad y se indique ejecutar la prueba. El ensayo hace diez ciclos de habilitación, espera `CLOCK_READY`, confirma lectura de control y siempre intenta dejar el reloj deshabilitado al terminar. No lee ni captura `DAT`.


In [ ]:
if RUN_CLOCK_EXERCISE:
    assert PROGRAM_PDM_CANDIDATE and PDM_OVERLAY is not None, "Cargar explícitamente el candidato antes del ejercicio de reloj"
    assert CLOCK_CYCLES == 10, "El criterio inicial de fase 2 usa diez ciclos"
    cycle_results = []
    try:
        for index in range(CLOCK_CYCLES):
            pdm.enable_clock()
            pdm.wait_clock_ready(timeout=CLOCK_READY_TIMEOUT_S)
            assert pdm.clock_enabled and pdm.clock_ready, "El reloj no alcanzó estado listo"
            enabled_state = {"clock_enabled": pdm.clock_enabled, "clock_ready": pdm.clock_ready}

            pdm.disable_clock()
            assert not pdm.clock_enabled, "CLOCK_ENABLE no se deshabilitó"
            cycle_results.append({"cycle": index + 1, "enabled": enabled_state,
                                  "disabled_clock_enable": pdm.clock_enabled,
                                  "disabled_clock_ready": pdm.clock_ready})
    finally:
        pdm.disable_clock()

    report["clock_exercise"] = {"cycles": cycle_results, "final_clock_enabled": pdm.clock_enabled}
    assert not pdm.clock_enabled, "El reloj debe quedar deshabilitado"
elif not PROGRAM_PDM_CANDIDATE:
    print("Prueba de reloj omitida: candidato no programado.")
else:
    print("Prueba de reloj omitida: RUN_CLOCK_EXERCISE=False")


## 4. Interpretación y límites

`PDM_CLOCK_HZ` y `CLOCK_READY` son valores nominales/estados internos, no una medición de frecuencia, duty ni calidad eléctrica en el pin. El multímetro sirve para continuidad y tensión DC, pero no certifica la forma de onda de 2,4 MHz. La conexión y comprobación física de micrófonos se hará cuando se indique; la lectura de cuatro canales requiere la captura RAW de fase 3.


In [ ]:
if PDM_OVERLAY is not None:
    report["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    report_path = run_dir / "acceptance-report.json"
    report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print("Informe:", report_path)
    print("No declarar aceptación física sin registrar las mediciones externas disponibles y sus límites.")
else:
    print("No se creó un informe de placa: el candidato solo se validó localmente.")
